# Optimal and Greedy Zero-Error Sensor Allocation

This notebook runs the exact and greedy algorithms on the complete 1D and 2D benchmarks. Each dataset contains 5,760 cases, so a full run evaluates 11,520 cases. There is no pilot mode or algorithm-selection mode.

The notebook reports a numerical summary and displays the three selected research figures.

In [1]:
from pathlib import Path
import sys

from IPython.display import Markdown, display

# Find the project folder when the notebook starts from either supported location.
candidates = [Path.cwd() / "min-sensor-zero-error", Path.cwd()]
PROJECT_DIRECTORY = next(
    candidate.resolve()
    for candidate in candidates
    if (candidate / "evaluate_benchmarks.py").is_file()
)

# Add the local project folder so its Python modules can be imported.
if str(PROJECT_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIRECTORY))

# Reload local modules when the notebook is restarted after code changes.
local_modules = (
    "optimal", "greedy", "phase_data", "field_models",
    "generate_inputs", "generate_benchmarks", "eval",
    "paper_analysis", "evaluate_benchmarks",
)
for module_name in local_modules:
    sys.modules.pop(module_name, None)

from evaluate_benchmarks import run_benchmark_evaluation

## Run settings

`REPEATS` affects runtime averaging only. `BOOTSTRAP_ITERATIONS` controls the confidence-interval calculation. Neither parameter changes which benchmark cases are evaluated.

In [2]:
# Number of deterministic runs used to average each runtime measurement.
REPEATS = 1

# Number of stratified bootstrap samples used for confidence intervals.
BOOTSTRAP_ITERATIONS = 2_000

print("Run mode: all 11,520 cases")
print("Runtime repetitions:", REPEATS)
print("Bootstrap iterations:", BOOTSTRAP_ITERATIONS)

Run mode: all 11,520 cases
Runtime repetitions: 1
Bootstrap iterations: 2000


## Run the complete evaluation

Missing benchmark inputs are generated automatically using fixed seeds. Existing inputs are reused. Both algorithms run on every case.

In [3]:
# Evaluate both dimensions and create the statistical analysis and figures.
analysis = run_benchmark_evaluation(
    repeats=REPEATS,
    bootstrap_iterations=BOOTSTRAP_ITERATIONS,
)

# Stop if the analysis does not contain exactly the selected figures.
SELECTED_FIGURES = ("families", "guarantee", "heatmaps")
if set(analysis["figures"]) != set(SELECTED_FIGURES):
    raise RuntimeError("The expected three-figure analysis was not created.")

Evaluating benchmark_1d ...
  5760 cases -> /Users/avishka/Academic/PhD/Research/QSN Research/Repos/quantum-sensor-placement-research/min-sensor-zero-error/results/algorithm_comparison/benchmark_1d_results_full/tables
Evaluating benchmark_2d ...
  5760 cases -> /Users/avishka/Academic/PhD/Research/QSN Research/Repos/quantum-sensor-placement-research/min-sensor-zero-error/results/algorithm_comparison/benchmark_2d_results_full/tables
Benchmark tables and figures -> /Users/avishka/Academic/PhD/Research/QSN Research/Repos/quantum-sensor-placement-research/min-sensor-zero-error/results/algorithm_comparison/results_summary_benchmark_1d_benchmark_2d_full


## Numerical summary

The first row combines both dimensions. The following rows report 1D and 2D separately.

In [4]:
# Format summary records as one Markdown table.
def summary_table(rows):
    header = (
        "| Group | Cases | Mean ratio | 95% CI | Exact-match rate | "
        "95th percentile | Maximum |\n"
        "|---|---:|---:|---:|---:|---:|---:|"
    )
    body = []
    for row in rows:
        body.append(
            f"| {row['group_value']} | {row['case_count']} | "
            f"{row['mean_greedy_ratio']:.3f} | "
            f"[{row['mean_greedy_ratio_ci95_low']:.3f}, "
            f"{row['mean_greedy_ratio_ci95_high']:.3f}] | "
            f"{100 * row['exact_match_rate']:.1f}% | "
            f"{row['p95_greedy_ratio']:.3f} | "
            f"{row['max_greedy_ratio']:.3f} |"
        )
    return header + "\n" + "\n".join(body)

summary_rows = [analysis["overall_summary"], *analysis["dimension_summaries"]]
display(Markdown(summary_table(summary_rows)))

| Group | Cases | Mean ratio | 95% CI | Exact-match rate | 95th percentile | Maximum |
|---|---:|---:|---:|---:|---:|---:|
| combined | 11520 | 1.152 | [1.149, 1.155] | 30.1% | 1.487 | 2.280 |
| 1D | 5760 | 1.161 | [1.157, 1.165] | 28.6% | 1.500 | 2.280 |
| 2D | 5760 | 1.144 | [1.141, 1.148] | 31.6% | 1.478 | 2.164 |

## Selected plots

1. **Sensor excess by field family:** compares Gaussian, sinusoidal, and quadratic fields across $K$.
2. **Approximation guarantee comparison:** shows how much of the theoretical sensor-excess allowance was used.
3. **Quality heatmaps:** shows mean greedy sensor excess across $K$ and $M$.

In [5]:
# Display only the three selected figures in the requested order.
plot_titles = {
    "families": "Sensor excess by field family",
    "guarantee": "Approximation guarantee comparison",
    "heatmaps": "Quality heatmaps",
}
for figure_key in SELECTED_FIGURES:
    display(Markdown(f"### {plot_titles[figure_key]}"))
    display(analysis["figures"][figure_key])

### Sensor excess by field family

<Figure size 1150x440 with 2 Axes>

### Approximation guarantee comparison

<Figure size 1150x420 with 2 Axes>

### Quality heatmaps

<Figure size 1150x480 with 3 Axes>

## Saved files

Detailed results are saved under `results/algorithm_comparison/<dataset>_results_full/tables/`. Combined tables and the three PNG/PDF figure pairs are saved under `results/algorithm_comparison/results_summary_benchmark_1d_benchmark_2d_full/`.